# Merging of the Bookings and Measurements datasets in respect to the overlapping timeframes in the column created_at

---

In [1]:
import pyarrow.parquet as pq
import duckdb
import os

In [2]:
base_path = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\filtered"
output_dir = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged"

filtered_measurements_file = os.path.join(base_path, "filtered_measurements_encoded.parquet")
filtered_bookings_file = os.path.join(base_path, "filtered_bookings.parquet")
merged_output_path = os.path.join(output_dir, "merged_bookings_measurements.parquet")

In [3]:
con = duckdb.connect()

### Load files into DuckDB as views (no data copied yet)

In [4]:
con.execute(f"CREATE OR REPLACE VIEW measurements AS SELECT * FROM '{filtered_measurements_file}';")
con.execute(f"CREATE OR REPLACE VIEW bookings AS SELECT * FROM '{filtered_bookings_file}';")

### Calculate overlapping time window, only load created_at column for efficiency

In [5]:
time_window = con.execute(f"""
    SELECT
        GREATEST(
            (SELECT MIN(created_at) FROM '{filtered_measurements_file}'),
            (SELECT MIN(created_at) FROM '{filtered_bookings_file}')
        ) AS start_time,
        LEAST(
            (SELECT MAX(created_at) FROM '{filtered_measurements_file}'),
            (SELECT MAX(created_at) FROM '{filtered_bookings_file}')
        ) AS end_time;
""").fetchdf()

start_time = time_window['start_time'][0]
end_time = time_window['end_time'][0]
print(f"Overlapping Time Window: {start_time} to {end_time}")

Overlapping Time Window: 2025-03-01 02:21:20.773000+01:00 to 2025-05-14 03:02:06.180000+02:00


### Filter datasets by time window and perform the merge

In [6]:
con.execute("""
    CREATE OR REPLACE TABLE merged AS
    SELECT
        m.* EXCLUDE (booking_id),
        b.*
    FROM measurements m
    INNER JOIN bookings b USING (booking_id)
    WHERE m.created_at BETWEEN ? AND ?
      AND b.created_at BETWEEN ? AND ?;
""", [start_time, end_time, start_time, end_time])

### Save merged dataset to Parquet

In [7]:
con.execute(f"COPY merged TO '{merged_output_path}' (FORMAT 'parquet');")

# Print summary
shape = con.execute("SELECT COUNT(*) AS rows FROM merged;").fetchdf()
print("Merged file saved to:", merged_output_path)
print("Merged shape rows:", shape['rows'][0])
# Print first 10 rows from the saved file
head_df = con.execute(f"SELECT * FROM '{merged_output_path}' LIMIT 10;").df()
print(head_df)

Merged file saved to: M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged\merged_bookings_measurements.parquet
Merged shape rows: 43934134
   measure_step_number  measure_value                       created_at  \
0                  888        558.232 2025-03-19 08:37:49.127000+01:00   
1                  888        554.270 2025-03-06 01:19:54.471000+01:00   
2                  888        551.225 2025-03-18 14:53:24.121000+01:00   
3                  888        561.035 2025-03-11 16:23:39.607000+01:00   
4                  888        560.873 2025-03-11 18:11:32.947000+01:00   
5                  888        558.825 2025-03-05 11:48:16.668000+01:00   
6                  888        554.432 2025-03-05 12:10:49.309000+01:00   
7                  888        560.523 2025-05-09 06:22:01.496000+02:00   
8                  888        556.992 2025-03-11 18:26:17.610000+01:00   
9                  888        554.755 2025-03-19 10:48:00.403000+01:00   

   book_s

In [8]:
# Close connection
con.close()